In [1]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('PEAKS_WGCNA_Module_Assignments.csv')

# Rename the 'Module' column to 'WGCNA_Module'
df = df.rename(columns={'Module': 'net.colors'})

# Save the modified dataframe back to CSV
df.to_csv('PEAKS_WGCNA_Module_Assignments.csv', index=False)

print("Successfully renamed 'Module' column to 'WGCNA_Module'")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")


Successfully renamed 'Module' column to 'WGCNA_Module'
Dataset shape: (18135, 2)
Columns: ['Peptide', 'net.colors']


In [1]:
# Fix WGCNA Module Assignments for GOparallel
# This script converts your module assignments to the proper format

import pandas as pd
import numpy as np
import re

# Set your file paths
module_file = "PEAKS_WGCNA_Module_Assignments-42M.csv"
peptide_file = "Peptide List Sweden Cohort.csv"
uniprot_file = "Copy of 2020-preference+2015_UniprotID-symbolLookupTableForR 3.xlsx"
output_file = "PEAKS_WGCNA_Module_assignments_GO-M42.csv"

# Read the files
print("Reading module assignments...")
modules = pd.read_csv(module_file)

print("Reading peptide list...")
peptides = pd.read_csv(peptide_file)

print("Reading UniProt mapping...")
uniprot = pd.read_excel(uniprot_file)

# Create gene name mapping from peptide list
peptide_to_gene = (peptides[['Peptide', 'Gene name', 'Accession']]
                   .dropna(subset=['Gene name'])
                   .query('`Gene name` != ""')
                   .drop_duplicates())

# Add UniProt mapping
uniprot_mapping = (uniprot.rename(columns={'Entry': 'Accession', 'Gene names  (primary )': 'Gene name'})
                   .dropna(subset=['Gene name']))

# Combine all mappings
all_accession_mapping = (pd.concat([peptide_to_gene[['Accession', 'Gene name']], 
                                   uniprot_mapping[['Accession', 'Gene name']]], ignore_index=True)
                         .drop_duplicates(subset=['Accession'], keep='first'))

# Process module assignments
print("Processing module assignments...")

# Extract peptide column (first column) and module colors
peptide_col = modules.columns[0]  # First column name
color_col = "net.colors"  # or whatever the module color column is named

# If the color column has a different name, find it
if color_col not in modules.columns:
    color_col = modules.columns[1]  # Use second column

# Create working data frame - keep ALL peptides including grey
working_modules = modules.rename(columns={peptide_col: 'peptide_id', color_col: 'module_color'}).copy()

# Function to reconstruct peptide ID in proper format
def reconstruct_peptide_id(row):
    peptide_id = row['peptide_id']
    
    if pd.isna(peptide_id):
        return peptide_id
    
    # Parse existing peptide ID
    parts = str(peptide_id).split('|')
    
    # Extract components
    if len(parts) >= 3:
        # Already in correct format
        gene_symbol = parts[0]
        accession = parts[1]
        peptide_seq = parts[2]
    elif len(parts) == 2:
        # Missing one component
        gene_symbol = parts[0]
        accession = parts[1]
        peptide_seq = ""
    else:
        # Only one component - assume it's gene symbol
        gene_symbol = parts[0]
        accession = ""
        peptide_seq = ""
    
    # Clean gene symbol
    gene_symbol = re.sub(r'\(.*?\)', '', gene_symbol)  # Remove modifications
    gene_symbol = gene_symbol.split(';')[0]  # Take first if semicolon-separated
    
    # Try to get gene name and accession from mappings if missing or unclear
    original_peptide = str(peptide_id)
    
    # Look up in peptide mapping first
    peptide_match = peptide_to_gene[peptide_to_gene['Peptide'] == original_peptide]
    if not peptide_match.empty:
        gene_symbol = peptide_match.iloc[0]['Gene name']
        if pd.notna(peptide_match.iloc[0]['Accession']):
            accession = peptide_match.iloc[0]['Accession']
    
    # If still missing accession, try to look up by gene symbol
    if not accession or accession == "":
        accession_match = all_accession_mapping[all_accession_mapping['Gene name'] == gene_symbol]
        if not accession_match.empty:
            accession = accession_match.iloc[0]['Accession']
    
    # If still missing gene symbol, try to look up by accession
    if accession and (not gene_symbol or gene_symbol == ""):
        gene_match = all_accession_mapping[all_accession_mapping['Accession'] == accession]
        if not gene_match.empty:
            gene_symbol = gene_match.iloc[0]['Gene name']
    
    # Reconstruct in proper format: GeneSymbol|AccessionID|PeptideSeq
    reconstructed = f"{gene_symbol}|{accession}|{peptide_seq}"
    
    return reconstructed

# Apply reconstruction to all peptides
print("Reconstructing peptide IDs...")
working_modules['reconstructed_id'] = working_modules.apply(reconstruct_peptide_id, axis=1)

# Create final output format - keep ALL peptides
output_data = (working_modules
               .dropna(subset=['reconstructed_id'])
               .query('reconstructed_id != ""')
               [['reconstructed_id', 'module_color']]
               .rename(columns={'reconstructed_id': 'Unique.ID', 'module_color': 'net.colors'}))

# Summary statistics
print("\n=== Processing Summary ===")
print(f"Total input peptides: {len(modules)}")
print(f"Successfully processed peptides: {len(output_data)}")
print(f"Total unique genes: {output_data['Unique.ID'].str.split('|').str[0].nunique()}")
print(f"Number of modules: {output_data['net.colors'].nunique()}")

# Module distribution
print("\nModule distribution:")
print(output_data['net.colors'].value_counts().sort_index())

# Show examples of reconstructed IDs
print("\nExamples of reconstructed peptide IDs:")
print(output_data['Unique.ID'].head(10).tolist())

# Write output
print(f"\nWriting fixed file to: {output_file}")
output_data.to_csv(output_file, index=False)

print(f"\n✅ Done! Use {output_file} with GOparallel.")
print(f"Maintained all {len(output_data)} peptide entries in proper format.")

# Show first few rows
print("\nFirst 10 rows of fixed file:")
print(output_data.head(10))

Reading module assignments...
Reading peptide list...


/var/folders/3z/7fq0vkpn29zf79cx6mwrk7s40000gn/T/ipykernel_95685/2844267974.py:19: DtypeWarning: Columns (138) have mixed types. Specify dtype option on import or set low_memory=False.
  peptides = pd.read_csv(peptide_file)


Reading UniProt mapping...
Processing module assignments...
Reconstructing peptide IDs...

=== Processing Summary ===
Total input peptides: 18135
Successfully processed peptides: 18135
Total unique genes: 1584
Number of modules: 43

Module distribution:
net.colors
black               261
blue                759
brown               686
cyan                111
darkgreen            63
darkgrey             55
darkmagenta          27
darkolivegreen       29
darkorange           49
darkred              68
darkturquoise        61
green               293
greenyellow         139
grey               6387
grey60               73
lightcyan            93
lightcyan1           16
lightgreen           71
lightsteelblue1      16
lightyellow          70
magenta             192
mediumpurple3        17
midnightblue        100
orange               50
orangered4           22
paleturquoise        39
pink                215
plum1                23
purple              159
red                 283
royalblue      